In [1]:
import re

from pathlib import Path

BASE_DIR = Path().cwd()
BASE_DIR

INPUT_FILE = BASE_DIR.joinpath("codigo_de_mineria_actualizado_2025_08_02.md")
OUTPUT_DIR = BASE_DIR.joinpath("data/titulos_codigo_mineria")

OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    content = f.read()

# Find all sections starting with '## TÍTULO' (including the section header)
# sections = re.split(r"(## TÍTULO [IVXLCDM]+)", content)
sections = [
    section
    for section in re.split(
        r"(## TÍTULO.*(?:\n|\r\n){2})|(## APÉNDICE.*(?:\n|\r\n){2})", content
    )
    if section is not None
]

# Extracts headers based on three patterns
titulos = [
    section
    for section in sections
    if re.match(r"(## TÍTULO FINAL)|(## TÍTULO ([IVXLCDM]+))|(## APÉNDICE)", section)
]
# Extracts the contents if they don't match the header patter or if they match the norm name at the top of the file.
contents = [
    section
    for section in sections
    if not (
        re.match(r"(## TÍTULO FINAL)|(## TÍTULO ([IVXLCDM]+))|(## APÉNDICE)", section)
        or section == "# CÓDIGO DE MINERÍA\n\n"
    )
]
# Generates a filename from the titulos: '## TÍTULO I: De las minas y su dominio\n\n' --> 'título_i.md'
filenames = [
    re.search(r"##\s*(.*?):", header).group(1).lower().replace(" ", "_") + ".md"
    for header in titulos
    if re.search(r"##\s*(.*?):", header)
]

for titulo, content, filename in zip(titulos, contents, filenames):
    with open(OUTPUT_DIR.joinpath(filename), "w", encoding="utf-8") as f:
        f.write(titulo + content)